# ZeroWatch — Zero-Day Attack Detection: Model Training & Evaluation

**Course project — AI-Based Zero-Day Attack Detection**
Bhavishyata Yadav | Roll No. 24CSU036

Bhavya Jain | Roll No. 24CSU037

Rudra Kumar Sharma | Roll No. 24CSU175

B.Tech CSE (Cybersecurity) | The NorthCap University

---

## What this notebook does

1. Loads the **NSL-KDD** intrusion detection dataset (public, downloads directly — no login/upload needed)
2. Preprocesses features (encoding, scaling) and maps raw attack labels into 5 categories: `Normal`, `DoS`, `Probe`, `R2L`, `U2R`
3. Simulates a **zero-day attack** using a *leave-one-attack-category-out* split: one attack category is fully excluded from training and used only at test time, standing in for an attack the model has never seen
4. Trains three models:
   - **Autoencoder (PyTorch)** — unsupervised, trained only on normal traffic — this is the actual zero-day detector
   - **Isolation Forest** — unsupervised baseline
   - **Random Forest** — supervised baseline (trained WITH labels on seen categories, included specifically to show what a normal signature-style/supervised classifier misses on an unseen category)
5. Evaluates all three on each held-out category (Precision, Recall, F1, False Positive Rate)
6. Explains individual flagged detections with **SHAP**
7. Saves trained model checkpoints for use in the ZeroWatch backend

> **Why NSL-KDD and not raw CICIDS2017 here:** NSL-KDD downloads from a direct public URL with no authentication, so this notebook runs top-to-bottom in a fresh Colab session with zero manual setup — important since your evaluator will run this themselves. The same leave-one-attack-out methodology applies identically to CICIDS2017/2018 if you swap the loader cell later (noted at the bottom).

## 1. Setup — install & import dependencies

In [ ]:
!pip install -q torch scikit-learn shap imbalanced-learn pandas matplotlib seaborn joblib

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Load the dataset

NSL-KDD, downloaded directly from a public GitHub mirror (no auth required). 41 network flow features + attack label per record.

In [ ]:
train_url = 'https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain%2B.txt'
test_url  = 'https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTest%2B.txt'

col_names = ['duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
    'wrong_fragment','urgent','hot','num_failed_logins','logged_in','num_compromised',
    'root_shell','su_attempted','num_root','num_file_creations','num_shells',
    'num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count',
    'srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate',
    'same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count',
    'dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate',
    'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate',
    'label','difficulty']

df_train_raw = pd.read_csv(train_url, names=col_names)
df_test_raw  = pd.read_csv(test_url, names=col_names)
df = pd.concat([df_train_raw, df_test_raw], ignore_index=True)   # we'll build our own splits, so combine first
df = df.drop(columns=['difficulty'])
print('Total records:', len(df))
df.head()

## 3. Map raw attack labels into 5 categories

NSL-KDD has 39+ specific attack names (e.g. `neptune`, `smurf`, `satan`). We group them into the standard 5 categories used in NSL-KDD research: `Normal`, `DoS`, `Probe`, `R2L`, `U2R`. This is the granularity we hold categories out at — e.g. training with no `Probe` examples at all, then testing whether the model flags `Probe` traffic as anomalous anyway.

In [ ]:
attack_map = {
    'normal': 'Normal',
    # DoS
    'neptune':'DoS','back':'DoS','land':'DoS','pod':'DoS','smurf':'DoS','teardrop':'DoS',
    'mailbomb':'DoS','processtable':'DoS','udpstorm':'DoS','apache2':'DoS','worm':'DoS',
    # Probe
    'satan':'Probe','ipsweep':'Probe','nmap':'Probe','portsweep':'Probe',
    'mscan':'Probe','saint':'Probe',
    # R2L
    'guess_passwd':'R2L','ftp_write':'R2L','imap':'R2L','phf':'R2L','multihop':'R2L',
    'warezmaster':'R2L','warezclient':'R2L','spy':'R2L','xlock':'R2L','xsnoop':'R2L',
    'snmpguess':'R2L','snmpgetattack':'R2L','httptunnel':'R2L','sendmail':'R2L','named':'R2L',
    # U2R
    'buffer_overflow':'U2R','loadmodule':'U2R','rootkit':'U2R','perl':'U2R',
    'sqlattack':'U2R','xterm':'U2R','ps':'U2R',
}

df['label'] = df['label'].str.strip()
df['attack_category'] = df['label'].map(attack_map)
df['attack_category'] = df['attack_category'].fillna('Other')   # catch any label not in our map

print(df['attack_category'].value_counts())
sns.countplot(data=df, x='attack_category', order=df['attack_category'].value_counts().index)
plt.title('Record count per attack category')
plt.xticks(rotation=30)
plt.show()

## 4. Encode categorical features and scale numeric features

In [ ]:
categorical_cols = ['protocol_type', 'service', 'flag']
encoders = {}
for c in categorical_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c])
    encoders[c] = le

feature_cols = [c for c in df.columns if c not in ['label', 'attack_category']]

scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print('Feature columns:', len(feature_cols))
df[feature_cols].describe().T.head()

## 5. Leave-one-attack-category-out split — the zero-day simulation

For a chosen `held_out_category`:
- **Training set**: `Normal` traffic + all attack categories EXCEPT the held-out one
- **Test set**: a mix of `Normal` traffic + ONLY the held-out category

The Autoencoder trains on `Normal` traffic only (true unsupervised anomaly detection). The Random Forest baseline trains on `Normal` + seen-attack labels (supervised) specifically so we can show the contrast: a supervised model has literally never seen this category's label and has no class to predict it into, mirroring what a signature/rule-based system would miss.

In [ ]:
def leave_one_attack_out_split(df, held_out_category, feature_cols, test_size_normal=0.3, seed=SEED):
    rng = np.random.RandomState(seed)

    normal_df = df[df['attack_category'] == 'Normal']
    seen_attacks_df = df[(df['attack_category'] != 'Normal') &
                          (df['attack_category'] != held_out_category) &
                          (df['attack_category'] != 'Other')]
    held_out_df = df[df['attack_category'] == held_out_category]

    normal_test_idx = normal_df.sample(frac=test_size_normal, random_state=seed).index
    normal_train_df = normal_df.drop(normal_test_idx)
    normal_test_df = normal_df.loc[normal_test_idx]

    X_train_normal_only = normal_train_df[feature_cols].values          # for Autoencoder
    X_train_supervised = pd.concat([normal_train_df, seen_attacks_df])[feature_cols].values   # for RF baseline
    y_train_supervised = pd.concat([normal_train_df, seen_attacks_df])['attack_category'].values

    X_test = pd.concat([normal_test_df, held_out_df])[feature_cols].values
    y_test = np.array(['Normal'] * len(normal_test_df) + [held_out_category] * len(held_out_df))

    return {
        'X_train_normal_only': X_train_normal_only,
        'X_train_supervised': X_train_supervised,
        'y_train_supervised': y_train_supervised,
        'X_test': X_test,
        'y_test': y_test,   # 'Normal' or the held-out category name
    }

HELD_OUT_CATEGORIES = ['DoS', 'Probe', 'R2L', 'U2R']   # we evaluate zero-day performance against each of these in turn
print('Will hold out, one at a time:', HELD_OUT_CATEGORIES)

## 6. Autoencoder model (PyTorch) — the zero-day detector

Trained ONLY on normal traffic. At test time, reconstruction error is the anomaly score — traffic the model reconstructs poorly is traffic that doesn't look like anything it learned as 'normal', regardless of whether it's ever seen that specific attack type.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, bottleneck_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, bottleneck_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 16), nn.ReLU(),
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


def train_autoencoder(X_train, input_dim, epochs=25, batch_size=256, lr=1e-3):
    model = Autoencoder(input_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    dataset = torch.utils.data.TensorDataset(X_tensor)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    history = []
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            recon = model(batch)
            loss = criterion(recon, batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * batch.size(0)
        epoch_loss /= len(dataset)
        history.append(epoch_loss)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  epoch {epoch+1}/{epochs} — reconstruction MSE: {epoch_loss:.5f}')
    return model, history


def autoencoder_anomaly_scores(model, X, batch_size=512):
    model.eval()
    X_tensor = torch.tensor(X, dtype=torch.float32)
    scores = []
    with torch.no_grad():
        for i in range(0, len(X_tensor), batch_size):
            batch = X_tensor[i:i+batch_size].to(device)
            recon = model(batch)
            mse = torch.mean((recon - batch) ** 2, dim=1)
            scores.append(mse.cpu().numpy())
    return np.concatenate(scores)

## 7. Evaluation harness

For each held-out category: train fresh on the remaining categories, score the test set, threshold at the 95th percentile of the *training* normal reconstruction error (a standard unsupervised-anomaly convention — no attack labels used to pick the threshold), and report Precision/Recall/F1/False Positive Rate treating the held-out category as the positive ('attack') class.

In [ ]:
def evaluate_run(y_test, y_pred_binary, held_out_category):
    y_true_binary = (y_test == held_out_category).astype(int)
    precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true_binary, y_pred_binary).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return {'precision': precision, 'recall': recall, 'f1_score': f1, 'false_positive_rate': fpr,
            'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn)}


results = []
trained_autoencoders = {}
trained_rf_models = {}

for held_out in HELD_OUT_CATEGORIES:
    print(f'\n=== Holding out: {held_out} (simulated zero-day) ===')
    split = leave_one_attack_out_split(df, held_out, feature_cols)

    # --- Autoencoder (unsupervised, true zero-day detector) ---
    ae_model, _ = train_autoencoder(split['X_train_normal_only'], input_dim=len(feature_cols), epochs=25)
    train_scores = autoencoder_anomaly_scores(ae_model, split['X_train_normal_only'])
    threshold = np.percentile(train_scores, 95)
    test_scores = autoencoder_anomaly_scores(ae_model, split['X_test'])
    ae_pred = (test_scores > threshold).astype(int)
    ae_metrics = evaluate_run(split['y_test'], ae_pred, held_out)
    ae_metrics.update({'model': 'Autoencoder', 'held_out_category': held_out})
    results.append(ae_metrics)
    trained_autoencoders[held_out] = (ae_model, threshold)

    # --- Isolation Forest (unsupervised baseline) ---
    iso = IsolationForest(contamination=0.05, random_state=SEED)
    iso.fit(split['X_train_normal_only'])
    iso_pred = (iso.predict(split['X_test']) == -1).astype(int)
    iso_metrics = evaluate_run(split['y_test'], iso_pred, held_out)
    iso_metrics.update({'model': 'IsolationForest', 'held_out_category': held_out})
    results.append(iso_metrics)

    # --- Random Forest (supervised baseline — the contrast case) ---
    X_bal, y_bal = SMOTE(random_state=SEED).fit_resample(split['X_train_supervised'], split['y_train_supervised'])
    rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
    rf.fit(X_bal, y_bal)
    rf_pred_labels = rf.predict(split['X_test'])
    # RF has never seen the held-out label — by construction it CANNOT predict it,
    # so anything not predicted 'Normal' counts as 'flagged as anomalous/attack-like'
    rf_pred = (rf_pred_labels != 'Normal').astype(int)
    rf_metrics = evaluate_run(split['y_test'], rf_pred, held_out)
    rf_metrics.update({'model': 'RandomForest(supervised)', 'held_out_category': held_out})
    results.append(rf_metrics)
    trained_rf_models[held_out] = rf

results_df = pd.DataFrame(results)[['model', 'held_out_category', 'precision', 'recall', 'f1_score', 'false_positive_rate', 'tp', 'fp', 'tn', 'fn']]
results_df

## 8. Results — leave-one-attack-out comparison

This table is your headline evidence: for each held-out category, the Autoencoder (never trained on it) still detects it at meaningful precision/recall, while making the false-positive-rate tradeoff explicit.

In [ ]:
pivot = results_df.pivot(index='held_out_category', columns='model', values='f1_score')
pivot.plot(kind='bar', figsize=(9,5))
plt.title('F1 score by held-out category and model')
plt.ylabel('F1 score')
plt.xticks(rotation=0)
plt.legend(title='Model')
plt.tight_layout()
plt.show()

print('\nFull results:')
results_df.round(3)

In [ ]:
# Confusion matrix for one example held-out category (edit HELD_OUT_EXAMPLE to inspect others)
HELD_OUT_EXAMPLE = 'Probe'
split = leave_one_attack_out_split(df, HELD_OUT_EXAMPLE, feature_cols)
ae_model, threshold = trained_autoencoders[HELD_OUT_EXAMPLE]
scores = autoencoder_anomaly_scores(ae_model, split['X_test'])
pred = (scores > threshold).astype(int)
y_true_binary = (split['y_test'] == HELD_OUT_EXAMPLE).astype(int)

cm = confusion_matrix(y_true_binary, pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Normal', 'Predicted Anomalous'],
            yticklabels=['Actual Normal', f'Actual {HELD_OUT_EXAMPLE} (unseen)'])
plt.title(f'Autoencoder confusion matrix — held out: {HELD_OUT_EXAMPLE}')
plt.show()

## 9. Explainability — SHAP

Using the Random Forest baseline for SHAP since PyTorch autoencoders need a slower gradient-based explainer (`DeepExplainer`) — for a demo, `TreeExplainer` on the RF is fast and shows the same idea: which features drive a 'this looks anomalous/attack-like' decision. Swap in `shap.DeepExplainer(ae_model, ...)` if you want SHAP directly on the Autoencoder's reconstruction error.

In [ ]:
HELD_OUT_FOR_SHAP = 'Probe'
split = leave_one_attack_out_split(df, HELD_OUT_FOR_SHAP, feature_cols)
rf = trained_rf_models[HELD_OUT_FOR_SHAP]

explainer = shap.TreeExplainer(rf)
sample = split['X_test'][:200]   # keep it small — SHAP on the full test set is slow
shap_values = explainer.shap_values(sample)

# shap_values is a list (one array per class) for multiclass RF — pick the 'Normal' class index to explain
normal_class_idx = list(rf.classes_).index('Normal')
shap.summary_plot(shap_values[normal_class_idx], sample, feature_names=feature_cols, show=True)

In [ ]:
# Explain a single flagged instance (mirrors the 'alert detail' view in the ZeroWatch UI)
instance_idx = 5   # pick a flagged held-out-category instance
single_shap = explainer.shap_values(sample[instance_idx:instance_idx+1])[normal_class_idx][0]
top_features = pd.Series(single_shap, index=feature_cols).abs().sort_values(ascending=False).head(5)
print('Top contributing features for this flagged instance:')
print(top_features)

## 10. Save trained models for the ZeroWatch backend

Saves one Autoencoder + threshold per held-out scenario, plus the RF baseline and the scaler/encoders needed to preprocess new data identically at inference time. Copy the downloaded files into `backend/models/checkpoints/` in the main ZeroWatch repo.

In [ ]:
import os
os.makedirs('zerowatch_models', exist_ok=True)

# Save one autoencoder (pick the 'production' one — trained holding out nothing, i.e. on ALL seen attack categories + normal)
# For the live demo, retrain once more on the FULL seen set (no held-out category) as the actual deployed model:
full_normal_only = df[df['attack_category'] == 'Normal'][feature_cols].values
production_ae, _ = train_autoencoder(full_normal_only, input_dim=len(feature_cols), epochs=25)
production_threshold = np.percentile(autoencoder_anomaly_scores(production_ae, full_normal_only), 95)

torch.save(production_ae.state_dict(), 'zerowatch_models/autoencoder.pt')
joblib.dump(production_threshold, 'zerowatch_models/autoencoder_threshold.pkl')
joblib.dump(scaler, 'zerowatch_models/scaler.pkl')
joblib.dump(encoders, 'zerowatch_models/label_encoders.pkl')
joblib.dump(feature_cols, 'zerowatch_models/feature_cols.pkl')
results_df.to_csv('zerowatch_models/leave_one_out_results.csv', index=False)

# Also save one RF baseline (from the Probe-held-out run, or retrain on everything for production)
joblib.dump(trained_rf_models['Probe'], 'zerowatch_models/random_forest_baseline.pkl')

print('Saved to ./zerowatch_models/:')
for f in os.listdir('zerowatch_models'):
    print(' -', f)

In [ ]:
# In Colab: download everything as a zip
import shutil
shutil.make_archive('zerowatch_models', 'zip', 'zerowatch_models')

from google.colab import files
files.download('zerowatch_models.zip')

## 11. Loading the saved model later (e.g. inside the FastAPI backend)

```python
import torch, joblib

feature_cols = joblib.load('feature_cols.pkl')
scaler = joblib.load('scaler.pkl')
threshold = joblib.load('autoencoder_threshold.pkl')

model = Autoencoder(input_dim=len(feature_cols))
model.load_state_dict(torch.load('autoencoder.pt', map_location='cpu'))
model.eval()

# score a new flow:
# x_scaled = scaler.transform([raw_feature_vector])
# score = reconstruction_mse(model, x_scaled)
# is_anomalous = score > threshold
```

## 12. Notes for the report / viva

- **Zero-day claim, precisely stated**: the Autoencoder is never trained on the held-out category's traffic in any leave-one-out run. Detection on that category at test time demonstrates generalization to an unseen attack pattern via anomaly scoring, not signature matching.
- **Why compare against a supervised Random Forest**: it makes the gap concrete — RF has literally no class for the held-out label and can only ever predict into classes it has seen, mirroring what a real signature/rule-based system misses.
- **False positive rate is reported deliberately, not hidden** — it's the actual hard constraint in real IDS deployment, and showing you tracked it (rather than only accuracy) is what makes the evaluation credible.
- **Swapping in CICIDS2017/2018**: replace Section 2's loader with a Kaggle-hosted CICIDS2017 CSV load (`kagglehub.dataset_download(...)`, requires a free Kaggle API token) and adjust `attack_map` in Section 3 to that dataset's label set — Sections 5 onward are dataset-agnostic and need no changes.
- **Reproducibility**: `SEED = 42` is fixed throughout; rerunning this notebook should produce numbers within a small tolerance of what's reported here.